# Gaming-Player Clustering — How Much Structure Is Really There?

### An unsupervised-learning case study on online-gaming behaviour

**Author:** Blagoy Hristov

---

## Table of contents

1. [Problem understanding, formulation and significance](#sec1)
2. [Mathematical formulation](#sec2)
3. [Data loading and sanity checks](#sec3)
4. [Exploratory data analysis](#sec4)
5. [Feature engineering and preprocessing](#sec5)
6. [Dimensionality and structure check with PCA](#sec6)
7. [K-Means clustering](#sec7)
8. [Choosing *k*: internal validation against a null model](#sec8)
9. [Alternative algorithms: GMM and DBSCAN](#sec9)
10. [External validation and cluster profiling](#sec10)
11. [Conclusions, limitations and future work](#sec11)
12. [References](#refs)

## How to reproduce

```bash
python -m pip install -r requirements.txt
jupyter lab   # then: Kernel ▸ Restart Kernel and Run All Cells
```

**Reproducibility conventions used throughout**

* Every stochastic step is seeded with `random_state = 42`.
* The dataset lives in `data/online_gaming_behavior_dataset.csv`; reusable
  helpers live in `src/clustering_utils.py`.
* The notebook is designed to run top-to-bottom with **Restart & Run All**;
  each cell depends only on cells above it.

<a id="sec3"></a>
## 3. Data loading and sanity checks

We load the dataset and run basic integrity checks (shape, types, missing
values, duplicates, and identifier uniqueness) before any analysis.

In [1]:
import sys
sys.path.insert(0, "src")  # make the reusable helper module importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

import clustering_utils as cu

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
cu.set_plot_style()

pd.set_option("display.max_columns", 20)
print("Libraries loaded. NumPy", np.__version__, "| pandas", pd.__version__)

Libraries loaded. NumPy 2.2.2 | pandas 2.2.3


In [2]:
df = pd.read_csv("data/online_gaming_behavior_dataset.csv")
print("Shape:", df.shape)
df.head()

Shape: (40034, 13)


,PlayerID,Age,Gender,Location,GameGenre,PlayTimeHours,InGamePurchases,GameDifficulty,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked,EngagementLevel
0,9000,43,Male,Other,Strategy,16.271119,0,Medium,6,108,79,25,Medium
1,9001,29,Female,USA,Strategy,5.525961,0,Medium,5,144,11,10,Medium
2,9002,22,Female,USA,Sports,8.223755,0,Easy,16,142,35,41,High
3,9003,35,Male,USA,Action,5.265351,1,Easy,9,85,57,47,Medium
4,9004,33,Male,Europe,Action,15.531945,0,Medium,2,131,95,37,Medium


In [3]:
# Column types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40034 entries, 0 to 40033
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   PlayerID                   40034 non-null  int64  
 1   Age                        40034 non-null  int64  
 2   Gender                     40034 non-null  object 
 3   Location                   40034 non-null  object 
 4   GameGenre                  40034 non-null  object 
 5   PlayTimeHours              40034 non-null  float64
 6   InGamePurchases            40034 non-null  int64  
 7   GameDifficulty             40034 non-null  object 
 8   SessionsPerWeek            40034 non-null  int64  
 9   AvgSessionDurationMinutes  40034 non-null  int64  
 10  PlayerLevel                40034 non-null  int64  
 11  AchievementsUnlocked       40034 non-null  int64  
 12  EngagementLevel            40034 non-null  object 
dtypes: float64(1), int64(7), object(5)
memory usag

In [4]:
# Integrity checks
print("Missing values per column:")
print(df.isna().sum().to_string())
print("\nDuplicate rows          :", int(df.duplicated().sum()))
print("Duplicate PlayerID values:", int(df["PlayerID"].duplicated().sum()))

Missing values per column:
PlayerID                     0
Age                          0
Gender                       0
Location                     0
GameGenre                    0
PlayTimeHours                0
InGamePurchases              0
GameDifficulty               0
SessionsPerWeek              0
AvgSessionDurationMinutes    0
PlayerLevel                  0
AchievementsUnlocked         0
EngagementLevel              0

Duplicate rows          : 0


Duplicate PlayerID values: 0


In [5]:
# Summary statistics of the numeric columns
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
PlayerID,40034.0,29016.50,11556.96,9000.0,19008.25,29016.50,39024.75,49033.0
Age,40034.0,31.99,10.04,15.0,23.00,32.00,41.00,49.0
PlayTimeHours,40034.0,12.02,6.91,0.0,6.07,12.01,17.96,24.0
InGamePurchases,40034.0,0.20,0.40,0.0,0.00,0.00,0.00,1.0
SessionsPerWeek,40034.0,9.47,5.76,0.0,4.00,9.00,14.00,19.0
AvgSessionDurationMinutes,40034.0,94.79,49.01,10.0,52.00,95.00,137.00,179.0
PlayerLevel,40034.0,49.66,28.59,1.0,25.00,49.00,74.00,99.0
AchievementsUnlocked,40034.0,24.53,14.43,0.0,12.00,25.00,37.00,49.0


**Sanity-check verdict.** The dataset is exceptionally clean: 40&nbsp;034 rows,
no missing values, no duplicate rows and a unique `PlayerID`. The `describe`
table already hints at the caveat from Section&nbsp;1 — every numeric feature
has a mean sitting almost exactly at the midpoint of its range (e.g. `Age`
$\approx 32$ on $[15,49]$, `PlayerLevel` $\approx 50$ on $[1,99]$), the signature
of a **uniform** distribution. We test this properly next.